In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install pretty_midi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.8 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=03ad7cf8e8258253c252c08058e2c0a7c724dceec5f7c966972f58f553d38b44
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [ ]:
import os

folders = [
    '/content/drive/MyDrive/music-project/outputs/generated_midis',
    '/content/drive/MyDrive/music-project/outputs/plots',
    '/content/drive/MyDrive/music-project/outputs/survey_results/tokens',
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("All folders created!")

All folders created!


In [ ]:
import zipfile, os

zip_path = '/content/drive/MyDrive/Music Project/maestro-v3.0.0-midi.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/maestro/')
print("CSV exists:", os.path.exists('/content/maestro/maestro-v3.0.0/maestro-v3.0.0.csv'))

CSV exists: True


In [ ]:
exec(open('/content/drive/MyDrive/Music Project/task1.py').read())
print("Task 1 code loaded!")

Training on device: cuda
CSV path:   /content/maestro/maestro-v3.0.0/maestro-v3.0.0.csv
MIDI root:  /content/maestro/maestro-v3.0.0
Output dir: /content/drive/MyDrive/music-project/outputs/generated_midis
Loading dataset...
[train] Loaded 62684 windows from 962 files
[validation] Loaded 7876 windows from 137 files
Parameters: 2,054,552
Epoch  10/50  Train: 0.1517  Val: 0.1578
Epoch  20/50  Train: 0.1035  Val: 0.1148
Epoch  30/50  Train: 0.0861  Val: 0.1033
Epoch  40/50  Train: 0.0781  Val: 0.0990
Epoch  50/50  Train: 0.0742  Val: 0.0990
Loss curve saved: /content/drive/MyDrive/music-project/outputs/plots/loss_curve_task1.png
Model weights saved: /content/drive/MyDrive/music-project/outputs/ae_weights.pt

Generating 5 MIDI samples...
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_1.mid  (33 notes, 8.0s)
  Sample 1: only 33 notes — try threshold=0.15
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_2.mid  (686 notes, 8.0s)

In [ ]:
import torch

CSV_PATH  = '/content/maestro/maestro-v3.0.0/maestro-v3.0.0.csv'
MIDI_ROOT = '/content/maestro/maestro-v3.0.0'
OUT_DIR   = '/content/drive/MyDrive/music-project/outputs/generated_midis'
PLOT_DIR  = '/content/drive/MyDrive/music-project/outputs/plots'
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {DEVICE}")

GPU available: True
Device: cuda


In [ ]:
from torch.utils.data import DataLoader

train_ds = MAESTRODataset(CSV_PATH, MIDI_ROOT, split='train')
val_ds   = MAESTRODataset(CSV_PATH, MIDI_ROOT, split='validation')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)

print("Dataset ready!")

[train] Loaded 62684 windows from 962 files
[validation] Loaded 7876 windows from 137 files
Dataset ready!


In [ ]:
LATENT = 64
EPOCHS = 50

model = LSTMAutoencoder(
    input_dim  = PIANO_KEYS,
    hidden_dim = 256,
    latent_dim = LATENT,
    seq_len    = SEQ_LEN
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Starting training...")

train_losses, val_losses = train(
    model, train_loader, val_loader,
    epochs=EPOCHS, device=DEVICE
)


torch.save(model.state_dict(),
           '/content/drive/MyDrive/music-project/outputs/ae_weights.pt')
print("Model saved to Drive!")

Model parameters: 2,054,552
Starting training...
Epoch  10/50  Train: 0.1763  Val: 0.1812
Epoch  20/50  Train: 0.1309  Val: 0.1431
Epoch  30/50  Train: 0.1030  Val: 0.1190
Epoch  40/50  Train: 0.0876  Val: 0.1051
Epoch  50/50  Train: 0.0792  Val: 0.1002
Model saved to Drive!


In [ ]:
plot_loss(train_losses, val_losses,
          save_path=f'{PLOT_DIR}/loss_curve_task1.png')
print("Plot saved to Drive!")

Loss curve saved: /content/drive/MyDrive/music-project/outputs/plots/loss_curve_task1.png
Plot saved to Drive!


In [ ]:
print("Generating 5 MIDI samples...")

for i in range(5):
    out_path = f'{OUT_DIR}/task1_sample_{i+1}.mid'
    n = generate_midi(
        model,
        latent_dim = LATENT,
        threshold  = 0.2,
        out_path   = out_path,
        device     = DEVICE
    )
    print(f"Sample {i+1}: {n} notes generated")

print("All 5 samples saved to Drive!")

Generating 5 MIDI samples...
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_1.mid  (76 notes, 8.0s)
Sample 1: 76 notes generated
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_2.mid  (146 notes, 8.0s)
Sample 2: 146 notes generated
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_3.mid  (36 notes, 8.0s)
Sample 3: 36 notes generated
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_4.mid  (155 notes, 8.0s)
Sample 4: 155 notes generated
Saved: /content/drive/MyDrive/music-project/outputs/generated_midis/task1_sample_5.mid  (20 notes, 8.0s)
Sample 5: 20 notes generated
All 5 samples saved to Drive!


In [ ]:
print("TASK 1 METRICS")
for i in range(1, 6):
    p = f'{OUT_DIR}/task1_sample_{i}.mid'
    if os.path.exists(p):
        rd = rhythm_diversity(p)
        rr = repetition_ratio(p)
        print(f"Sample {i}: Rhythm Diversity = {rd:.3f} | Repetition Ratio = {rr:.3f}")

TASK 1 METRICS
Sample 1: Rhythm Diversity = 0.237 | Repetition Ratio = 0.027
Sample 2: Rhythm Diversity = 0.205 | Repetition Ratio = 0.000
Sample 3: Rhythm Diversity = 0.333 | Repetition Ratio = 0.000
Sample 4: Rhythm Diversity = 0.187 | Repetition Ratio = 0.000
Sample 5: Rhythm Diversity = 0.450 | Repetition Ratio = 0.000
